# 01 · Datos y análisis exploratorio

Entrada: `processed.cleveland.data` (UCI) y, opcionalmente, `heart.csv`.
Salida: `datos_cleveland_limpios.csv` (con perdidos) y tablas y figuras en `figures/`.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
SEMILLA = 42
os.makedirs("figures", exist_ok=True)

## 1. Carga y control de calidad

In [ ]:
try:
    from google.colab import files
    if not os.path.exists("processed.cleveland.data"):
        files.upload()
except ImportError:
    pass

COLUMNAS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
            "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
df = pd.read_csv("processed.cleveland.data", header=None, names=COLUMNAS, na_values="?")
df.shape

In [ ]:
print(df.isna().sum()[df.isna().sum() > 0])
print("Filas con perdidos:", df.isna().any(axis=1).sum(), "| duplicadas:", df.duplicated().sum())
df[df.isna().any(axis=1)]

In [ ]:
df.describe().T[["min", "max", "mean"]].round(1)

Comparación con `heart.csv`, que trae los perdidos ya imputados.

In [ ]:
if os.path.exists("heart.csv"):
    heart = pd.read_csv("heart.csv")
    distintas = df.fillna(-1).values != heart.values
    filas = np.where(distintas.any(axis=1))[0]
    print("Celdas distintas:", distintas.sum(), "| filas:", filas.tolist())
    display(pd.concat({"UCI": df.loc[filas, ["ca", "thal"]],
                       "heart.csv": heart.loc[filas, ["ca", "thal"]]}, axis=1))

## 2. Desenlace y recodificación

In [ ]:
df["enfermedad"] = (df["num"] >= 1).astype(int)      # estenosis > 50 %
df["restecg_anormal"] = (df["restecg"] >= 1).astype(int)  # categoría 1: n = 4
print(df["num"].value_counts().sort_index().to_dict())
print(f"Eventos: {df['enfermedad'].sum()}/{len(df)} ({df['enfermedad'].mean():.1%})")

## 3. Tabla 1

In [ ]:
CONTINUAS = {"age": "Edad, años", "trestbps": "PA sistólica, mmHg",
             "chol": "Colesterol, mg/dl", "thalach": "FC máxima, lpm",
             "oldpeak": "Descenso ST, mm"}
CATEGORICAS = {
    "sex": ("Sexo", {0: "Mujer", 1: "Hombre"}),
    "cp": ("Tipo de dolor", {1: "Angina típica", 2: "Angina atípica",
                             3: "Dolor no anginoso", 4: "Asintomático"}),
    "fbs": ("Glucemia > 120 mg/dl", {0: "No", 1: "Sí"}),
    "restecg_anormal": ("ECG en reposo", {0: "Normal", 1: "Anormal"}),
    "exang": ("Angina con el ejercicio", {0: "No", 1: "Sí"}),
    "slope": ("Pendiente del ST", {1: "Ascendente", 2: "Plana", 3: "Descendente"}),
    "ca": ("Vasos coloreados", {0: "0", 1: "1", 2: "2", 3: "3"}),
    "thal": ("Talio", {3: "Normal", 6: "Defecto fijo", 7: "Defecto reversible"}),
}

def tabla_1(datos, grupo, etiquetas_grupo):
    """Media ± DE o mediana [RIC] según Shapiro; Welch / Mann-Whitney; χ²."""
    g0, g1 = [datos[datos[grupo] == k] for k in (0, 1)]
    filas = []
    for var, nombre in CONTINUAS.items():
        a, b, tot = g0[var].dropna(), g1[var].dropna(), datos[var].dropna()
        if stats.shapiro(a).pvalue > 0.05 and stats.shapiro(b).pvalue > 0.05:
            f = lambda x: f"{x.mean():.1f} ± {x.std():.1f}"
            p = stats.ttest_ind(a, b, equal_var=False).pvalue
        else:
            f = lambda x: f"{x.median():.1f} [{x.quantile(.25):.1f}–{x.quantile(.75):.1f}]"
            p = stats.mannwhitneyu(a, b).pvalue
        filas.append([nombre, f(tot), f(a), f(b), p])
    for var, (nombre, codigos) in CATEGORICAS.items():
        tabla = pd.crosstab(datos[var], datos[grupo])
        p = stats.chi2_contingency(tabla).pvalue if tabla.shape[0] > 1 else np.nan
        filas.append([nombre, "", "", "", p])
        for cod, txt in codigos.items():
            n_tot = (datos[var] == cod).sum()
            n0, n1 = (g0[var] == cod).sum(), (g1[var] == cod).sum()
            filas.append([f"   {txt}", f"{n_tot} ({n_tot/datos[var].notna().sum():.0%})",
                          f"{n0} ({n0/g0[var].notna().sum():.0%})",
                          f"{n1} ({n1/g1[var].notna().sum():.0%})", np.nan])
        if datos[var].isna().any():
            filas.append(["   Perdidos", str(datos[var].isna().sum()),
                          str(g0[var].isna().sum()), str(g1[var].isna().sum()), np.nan])
    cab = ["Variable", f"Total (n = {len(datos)})",
           f"{etiquetas_grupo[0]} (n = {len(g0)})", f"{etiquetas_grupo[1]} (n = {len(g1)})", "p"]
    t = pd.DataFrame(filas, columns=cab)
    t["p"] = t["p"].apply(lambda x: "" if pd.isna(x) else ("< 0.001" if x < 0.001 else f"{x:.3f}"))
    for col in t.columns[1:]:
        t[col] = t[col].str.replace(".", ",", regex=False)
    return t

t1 = tabla_1(df, "enfermedad", ("Sin enfermedad", "Con enfermedad"))
t1.to_csv("figures/tabla1_por_desenlace.csv", index=False)
t1

In [ ]:
t1_sexo = tabla_1(df, "sex", ("Mujeres", "Hombres"))
t1_sexo = t1_sexo[~t1_sexo["Variable"].str.strip().isin(["Sexo", "Mujer", "Hombre"])]
t1_sexo.to_csv("figures/tabla1_por_sexo.csv", index=False)
t1_sexo

## 4. Figuras

In [ ]:
dolor = df.groupby("cp")["enfermedad"].agg(n="count", con_enfermedad="sum", prevalencia="mean")
dolor.index = dolor.index.map(CATEGORICAS["cp"][1])
dolor["prevalencia"] = (dolor["prevalencia"] * 100).round(1)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(dolor.index, dolor["prevalencia"], color="#4C72B0")
for i, (v, n) in enumerate(zip(dolor["prevalencia"], dolor["n"])):
    ax.text(i, v + 1.5, f"{v:.0f} %\n(n = {n})", ha="center", fontsize=9)
ax.set_ylabel("Pacientes con enfermedad (%)")
ax.set_ylim(0, 100)
ax.set_title("Prevalencia de enfermedad coronaria según el tipo de dolor")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("figures/fig_prevalencia_por_dolor.png", dpi=300)
plt.show()
dolor

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, (var, nombre) in zip(axes, CONTINUAS.items()):
    ax.boxplot([df.loc[df["enfermedad"] == k, var].dropna() for k in (0, 1)], widths=0.6)
    ax.set_xticks([1, 2], ["Sin enfermedad", "Con enfermedad"], fontsize=8)
    ax.set_title(nombre, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Variables continuas según enfermedad coronaria", y=1.02)
plt.tight_layout()
plt.savefig("figures/fig_continuas_por_desenlace.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
corr = df[list(CONTINUAS) + ["ca"]].corr(method="spearman")
etiquetas = [CONTINUAS.get(c, "Vasos coloreados").split(",")[0] for c in corr.columns]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), etiquetas, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), etiquetas)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="ρ de Spearman")
plt.tight_layout()
plt.savefig("figures/fig_correlacion_spearman.png", dpi=300)
plt.show()

## 5. Exportación

In [ ]:
df.to_csv("datos_cleveland_limpios.csv", index=False)   # perdidos sin imputar
try:
    from google.colab import files
    files.download("datos_cleveland_limpios.csv")
except ImportError:
    pass